In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import csv
import os
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
tweets = pd.read_csv('/content/drive/MyDrive/labeled_tweets.csv')

In [ ]:
tweets.head()

,text,date,stock_label
0,"...For years, I watched one betrayal after ano...",2020-10-24 02:06:10+00:00,dollar
1,RT @ArthurSchwartz: Texas Lt. Gov. Dan Patrick...,2020-11-11 02:44:06+00:00,dollar
2,European Countries are sadly getting clobbered...,2020-11-16 16:11:09+00:00,euro
3,RT @DonaldJTrumpJr: 🚨🚨🚨Hunter Biden Offered $1...,2020-10-16 04:37:02+00:00,dollar
4,$13.9M is heading to New Orleans in @USDOT fun...,2020-08-12 18:43:05+00:00,dollar


In [5]:
print(len(tweets))

2021


In [ ]:
!pip install -q transformers torch scipy

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
from scipy.special import softmax
from tqdm.notebook import tqdm

# Wczytanie modelu sentiment analysis trenowanego na tweetach
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

# Funkcja przypisująca predykcję na podstawie sentymentu
def predict_effect(text):
    encoded_input = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        output = model(**encoded_input)
    scores = output.logits[0].numpy()
    scores = softmax(scores)

    # Miejsca: [negative, neutral, positive]
    max_idx = np.argmax(scores)
    if max_idx == 0:
        return 'decrease'
    elif max_idx == 1:
        return 'no change'
    else:
        return 'increase'

# Dodanie kolumny 'predicted_effect' do Twojej tabeli 'tweets'
tqdm.pandas(desc="Analiza tweetów")
tweets['predicted_effect'] = tweets['text'].progress_apply(predict_effect)

# Podgląd wyników
print(tweets[['text', 'stock_label', 'predicted_effect']].head(10))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.8 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Analiza tweetów:   0%|          | 0/2021 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

                                                text stock_label  \
0  ...For years, I watched one betrayal after ano...      dollar   
1  RT @ArthurSchwartz: Texas Lt. Gov. Dan Patrick...      dollar   
2  European Countries are sadly getting clobbered...        euro   
3  RT @DonaldJTrumpJr: 🚨🚨🚨Hunter Biden Offered $1...      dollar   
4  $13.9M is heading to New Orleans in @USDOT fun...      dollar   
5  GREAT news! New government of Sudan, which is ...      dollar   
6  RT @iheartmindy: Dominion Voting System donate...      dollar   
7  RT @AdamMilstein: Fred Trump sold property to ...      dollar   
8  But 2020 is a long way from over! https://t.co...      dollar   
9  RT @WhiteHouse: The Trump Administration stand...      dollar   

  predicted_effect  
0         decrease  
1        no change  
2         decrease  
3        no change  
4         increase  
5         increase  
6        no change  
7        no change  
8        no change  
9        no change  


In [ ]:
tweets.to_csv('/content/drive/MyDrive/tweets_with_predictions.csv', index=False)

In [16]:
tweets = pd.read_csv('/content/drive/MyDrive/tweets_with_predictions.csv')

In [4]:
tweets.head()

,text,date,stock_label,predicted_effect
0,"...For years, I watched one betrayal after ano...",2020-10-24 02:06:10+00:00,dollar,decrease
1,RT @ArthurSchwartz: Texas Lt. Gov. Dan Patrick...,2020-11-11 02:44:06+00:00,dollar,no change
2,European Countries are sadly getting clobbered...,2020-11-16 16:11:09+00:00,euro,decrease
3,RT @DonaldJTrumpJr: 🚨🚨🚨Hunter Biden Offered $1...,2020-10-16 04:37:02+00:00,dollar,no change
4,$13.9M is heading to New Orleans in @USDOT fun...,2020-08-12 18:43:05+00:00,dollar,increase


In [42]:
from datetime import datetime

# 1. Ścieżki do danych
BASE_PATHS = {
    'euro': r'/content/drive/MyDrive/THESIS/EUR_USD/all',
    'sp500': r'/content/drive/MyDrive/THESIS/SPX500_USD/all',
    'dollar': r'/content/drive/MyDrive/THESIS/EUR_USD/all',  # tymczasowo EUR/USD
}


In [44]:
# 2. Wczytaj tweety
tweets = pd.read_csv('/content/drive/MyDrive/tweets_with_predictions.csv', parse_dates=["date"])

In [43]:
# 3. Funkcja do pobierania ceny z danych rynkowych
def get_price_at_tweet(row):
    label = row['stock_label']
    timestamp = row['date']

    if pd.isna(label) or pd.isna(timestamp):
        return None

    year = timestamp.year
    path = os.path.join(BASE_PATHS[label], f"{year}.csv")

    if not os.path.exists(path):
        print(f"Brak danych dla {label} w roku {year}")
        return None

    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"Błąd wczytywania {path}: {e}")
        return None

    # Dopasuj nazwę kolumny z czasem
    time_col = 'time' if 'time' in df.columns else 'datetime'

    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df = df.set_index(time_col)

    # Znajdź dokładną minutę
    rounded_time = timestamp.replace(second=0, microsecond=0)

    if rounded_time in df.index:
        return df.loc[rounded_time]['close']
    else:
        return None


In [46]:
# Ogranicz tweetów tylko do lat 2019 i 2020
tweets = tweets[tweets['date'].dt.year.isin([2019])].copy()


In [47]:
print(f"Liczba wierszy w dataframe tweets: {len(tweets)}")


Liczba wierszy w dataframe tweets: 283


In [35]:
path = "/content/drive/MyDrive/THESIS/EUR_USD/all/2020.csv"
try:
    df_test = pd.read_csv(path, engine='python')
    print("OK! Plik da się wczytać.")
except Exception as e:
    print(f"Plik uszkodzony lub błędny: {e}")


OK! Plik da się wczytać.


In [48]:
def get_price_at_tweet(row):
    label = row['stock_label']
    timestamp = row['date']

    if pd.isna(label) or pd.isna(timestamp):
        return None

    year = timestamp.year
    path = os.path.join(BASE_PATHS[label], f"{year}.csv")

    if not os.path.exists(path):
        print(f"Brak danych dla {label} w roku {year}")
        return None

    try:
        df = pd.read_csv(path, engine='python')  # bezpieczny parser
    except Exception as e:
        print(f"Błąd wczytywania {path}: {e}")
        return None

    # Dopasuj kolumnę z czasem
    time_col = 'time' if 'time' in df.columns else 'datetime'
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df = df.set_index(time_col)
    df = df.sort_index()

    # Zaokrąglij do minuty (usuń sekundy i mikrosekundy)
    rounded_time = timestamp.replace(second=0, microsecond=0).tz_localize(None)


    # Znajdź dokładnie tę minutę lub najbliższą późniejszą
    if rounded_time in df.index:
        return df.loc[rounded_time]['close']
    else:
        later = df[df.index >= rounded_time]
        if not later.empty:
            return later.iloc[0]['close']
        else:
            return None


In [56]:
tweets = tweets.head(5)

In [57]:
tweets.head()

,text,date,stock_label,predicted_effect,price_at_tweet
91,"At my meeting with Jay Powell this morning, I ...",2019-11-19 03:34:12+00:00,dollar,decrease,1.10744
92,"Just finished a very good &amp, cordial meetin...",2019-11-18 16:01:42+00:00,dollar,increase,1.10823
93,"Republicans &amp, others must remember, the Uk...",2019-11-17 20:08:08+00:00,euro,decrease,1.10528
104,I will be signing our 738 Billion Dollar Defen...,2019-12-20 14:19:20+00:00,dollar,increase,NaN
107,RT @RepKenBuck: The President put America firs...,2019-12-12 17:54:06+00:00,dollar,no change,NaN


In [26]:
# Sprawdź 1 konkretny tweet z 2019 lub 2020 roku
example = tweets.iloc[0]  # weź pierwszy tweet (możesz podmienić na inny)

price = get_price_at_tweet(example)

print("Tweet:")
print(example[['text', 'stock_label', 'date']])
print("\nCena rynkowa przypisana do tego tweeta:", price)


Tweet:
text           At my meeting with Jay Powell this morning, I ...
stock_label                                               dollar
date                                   2019-11-19 03:34:12+00:00
Name: 91, dtype: object

Cena rynkowa przypisana do tego tweeta: 1.10744


In [58]:
# Przetestuj i pokaż 3 pierwsze tweety z przypisaną ceną
for i in range(3):
    row = tweets.iloc[i]
    price = get_price_at_tweet(row)
    tweets.at[row.name, 'price_at_tweet'] = price  # <- poprawnie według indeksu

# Wyświetl jako tabelę
display(tweets.iloc[:3][['date','text', 'stock_label','predicted_effect', 'price_at_tweet']])



,date,text,stock_label,predicted_effect,price_at_tweet
91,2019-11-19 03:34:12+00:00,"At my meeting with Jay Powell this morning, I ...",dollar,decrease,1.10744
92,2019-11-18 16:01:42+00:00,"Just finished a very good &amp, cordial meetin...",dollar,increase,1.10823
93,2019-11-17 20:08:08+00:00,"Republicans &amp, others must remember, the Uk...",euro,decrease,1.10528


In [53]:
def get_price_info_at_tweet(row):
    label = row['stock_label']
    timestamp = row['date']

    if pd.isna(label) or pd.isna(timestamp):
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    year = timestamp.year
    path = os.path.join(BASE_PATHS[label], f"{year}.csv")

    if not os.path.exists(path):
        print(f"Brak danych dla {label} w roku {year}")
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    try:
        df = pd.read_csv(path, engine='python')
    except Exception as e:
        print(f"Błąd wczytywania {path}: {e}")
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    time_col = 'time' if 'time' in df.columns else 'datetime'
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df = df.set_index(time_col).sort_index()

    # Ustaw timestamps (bez strefy czasowej)
    rounded = timestamp.replace(second=0, microsecond=0).tz_localize(None)
    times = {
        'price_now': rounded,
        'price_5min': rounded + pd.Timedelta(minutes=5),
        'price_10min': rounded + pd.Timedelta(minutes=10),
        'price_1h': rounded + pd.Timedelta(hours=1)
    }

    # Szukaj najbliższej późniejszej wartości
    result = []
    for label, t in times.items():
        if t in df.index:
            result.append(df.loc[t]['close'])
        else:
            later = df[df.index >= t]
            if not later.empty:
                result.append(later.iloc[0]['close'])
            else:
                result.append(None)

    return pd.Series(result, index=['price_now', 'price_5min', 'price_10min', 'price_1h'])


In [60]:
for i in range(3):
    row = tweets.iloc[i]
    prices = get_price_info_at_tweet(row)
    for col in prices.index:
        tweets.at[row.name, col] = prices[col]

display(tweets.iloc[:3][['date', 'text', 'stock_label', 'predicted_effect', 'price_now', 'price_5min', 'price_10min', 'price_1h']])


,date,text,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h
91,2019-11-19 03:34:12+00:00,"At my meeting with Jay Powell this morning, I ...",dollar,decrease,1.10744,1.10744,1.10745,1.10750
92,2019-11-18 16:01:42+00:00,"Just finished a very good &amp, cordial meetin...",dollar,increase,1.10823,1.10826,1.10868,1.10797
93,2019-11-17 20:08:08+00:00,"Republicans &amp, others must remember, the Uk...",euro,decrease,1.10528,1.10528,1.10528,1.10528


In [ ]:
# 5. Zapisz efekt do nowego pliku CSV
tweets.to_csv("tweets_with_market_price.csv", index=False)